# Phase 3 — Step 8 (rebuild): L6 IT vs L6 CT

**Why this notebook is the rebuild.** The v1 notebook averaged A1 / C1 / D1 features across hashes (forbidden by `WORKFLOW.md §3.3`). This rebuild uses the long-row + per-neuron probability aggregation protocol (`WORKFLOW.md §3.6`) and the `scan-only` LOSO integrity check from notebook 03 v2.

**Population.** `subtype ∈ {6P-IT, 6P-CT}` on the Phase-1 working population. From Step 1: 6P-IT n=216, 6P-CT n=121, 337 cells across 7 scans. **6 valid LOSO scans** (4_7 has only 1 cell so it's invalid; 9_*, 8_5, 7_3, 7_5 have ~zero L6 cells).

**PHASE3_PLANNING §3 commitment for L6.** L6 cells are deeper, often noisier, and L6 IT vs CT has known soma-size differences that can plausibly translate to SNR differences. We therefore report **cc_abs-residualized LOSO valid bal_acc as the *primary* headline** for L6, not as a sensitivity analysis. Both numbers are reported, but the residualized one is the one that matters.

We run **GKF and LOSO in this single notebook** because the L6 population is small (n=337) and the LOSO valid list is tight (6 scans).


## 1. Setup

In [1]:
from __future__ import annotations

import sys, time, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path('..').resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)
warnings.filterwarnings('ignore', category=FutureWarning, module='sklearn')

from src.config import (PROCESSED_TABLES_DIR, PROCESSED_FEATURES_DIR,
                        PROCESSED_RESULTS_DIR, RANDOM_SEED, ensure_dirs)
from src.data.loaders import build_modeling_table
from src.eval.metrics import neuron_level_score, summarize_cv_runs
from src.features.tier_b import B_FEATURE_NAMES
from src.features.tier_c import C1_FEATURE_NAMES
from src.features.tier_d import D_FEATURE_NAMES

ensure_dirs()
np.random.seed(RANDOM_SEED)

DATA_TABLES   = REPO_ROOT / 'data' / 'processed' / 'tables'
DATA_FEATURES = REPO_ROOT / 'data' / 'processed' / 'features'
DATA_RESULTS  = REPO_ROOT / 'data' / 'processed' / 'results'

LOSO_VALID_PATH = DATA_TABLES / 'phase3_loso_valid_scans.json'
GKF_OUT     = DATA_RESULTS / 'phase3_l6_it_ct_runs.parquet'
LOSO_OUT    = DATA_RESULTS / 'phase3_l6_it_ct_loso.parquet'
PERSCAN_OUT = DATA_RESULTS / 'phase3_l6_it_ct_loso_perscan.parquet'
WINNER_OUT  = DATA_RESULTS / 'phase3_l6_it_ct_winner.json'

CLASSES = np.array(['6P-IT', '6P-CT'])  # IT majority, CT minority
N_FOLDS = 5


## 2. Build long-row table, restrict to L6 IT/CT, refresh folds

In [2]:
# 2.1 A1 long via canonical loader
X_a1, y_a1, g_a1, f_a1, df_a1 = build_modeling_table(
    level='A1', blocks=['amp', 'shape'], label='celltype_label')

# 2.2 + B (inner join restricts to repeated hashes)
b_hash = pd.read_parquet(DATA_FEATURES / 'B_per_hash.parquet',
    columns=['nucleus_id','condition_hash'] + list(B_FEATURE_NAMES))
df_a1b = df_a1.merge(b_hash, on=['nucleus_id','condition_hash'],
                     how='inner', validate='one_to_one')

# 2.3 + C1
c1 = pd.read_parquet(DATA_FEATURES / 'C1_per_hash.parquet',
    columns=['nucleus_id','condition_hash'] + list(C1_FEATURE_NAMES))
df_a1bc1 = df_a1b.merge(c1, on=['nucleus_id','condition_hash'],
                        how='left', validate='one_to_one')

# 2.4 + D
d = pd.read_parquet(DATA_FEATURES / 'D_per_hash.parquet',
    columns=['condition_hash'] + list(D_FEATURE_NAMES))
df_full = df_a1bc1.merge(d, on='condition_hash', how='left', validate='many_to_one')

# 2.5 + G broadcast
G = pd.read_parquet(DATA_FEATURES / 'G_per_neuron.parquet')
g_cols = [c for c in G.columns if c.startswith('g_')]
df_full = df_full.merge(G[['nucleus_id'] + g_cols], on='nucleus_id',
                        how='left', validate='many_to_one')

# 2.6 Restrict to L6 IT/CT
df = df_full[df_full['celltype_label'].isin(['6P-IT','6P-CT'])].copy().reset_index(drop=True)
print(f'L6 IT/CT long-row table: {df.shape}, '
      f'{df["nucleus_id"].nunique()} neurons, {df["session_key"].nunique()} scans')
print()
print('class counts (neurons):')
print(df.drop_duplicates('nucleus_id')['celltype_label'].value_counts().to_string())
print()
print('per-scan crosstab (neurons):')
nd = df.drop_duplicates('nucleus_id')
print(pd.crosstab(nd['session_key'], nd['celltype_label']).to_string())
print()
print('cc_abs distribution by class (mean, std):')
print(nd.groupby('celltype_label')['cc_abs'].agg(['mean','std','median','count']).round(3).to_string())

# 2.7 Refresh GKF folds stratified by celltype_label, grouped by nucleus_id
neu = df[['nucleus_id','celltype_label','session_key']].drop_duplicates('nucleus_id').reset_index(drop=True)
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
neu['gkf_fold_phase3'] = -1
for k, (_, te) in enumerate(sgkf.split(np.zeros((len(neu),1)),
                                       neu['celltype_label'].to_numpy(),
                                       neu['nucleus_id'].to_numpy())):
    neu.loc[te,'gkf_fold_phase3'] = k
df = df.merge(neu[['nucleus_id','gkf_fold_phase3']], on='nucleus_id',
              how='left', validate='many_to_one')
print('\nper-fold neuron counts:')
print(neu.groupby(['gkf_fold_phase3','celltype_label']).size().unstack(fill_value=0).to_string())

# 2.8 LOSO validity (locked from Step 1)
with open(LOSO_VALID_PATH) as f:
    loso_meta = json.load(f)
VALID_SCANS = list(loso_meta['L6_IT_vs_CT']['valid_scans'])
INVALID_SCANS = list(loso_meta['L6_IT_vs_CT']['invalid_scans'])
print(f'\nLOSO validity rule: {loso_meta["rule"]}')
print(f'valid scans   ({len(VALID_SCANS)}):', VALID_SCANS)
print(f'invalid scans ({len(INVALID_SCANS)}):', INVALID_SCANS)


L6 IT/CT long-row table: (45832, 168), 337 neurons, 7 scans

class counts (neurons):
celltype_label
6P-IT    216
6P-CT    121

per-scan crosstab (neurons):
celltype_label  6P-CT  6P-IT
session_key                 
4_7                 1      0
5_6                50     46
5_7                24     57
6_2                 9     27
6_4                14     41
6_6                11     23
6_7                12     22

cc_abs distribution by class (mean, std):
                 mean    std  median  count
celltype_label                             
6P-CT           0.284  0.158   0.262    121
6P-IT           0.317  0.158   0.313    216

per-fold neuron counts:
celltype_label   6P-CT  6P-IT
gkf_fold_phase3              
0                   26     42
1                   23     44
2                   23     44
3                   29     39
4                   20     47

LOSO validity rule: {'min_minority_cells': 5, 'both_classes_required': True, 'frozen_at_step': 'phase3_step1'}
valid scans   (6)

## 3. Define feature blocks

In [3]:
amp_cols   = [c for c in df.columns if c.startswith('amp_')]
shape_cols = [c for c in df.columns if c.startswith('shape_')]
a1_cols    = amp_cols + shape_cols
b_cols     = list(B_FEATURE_NAMES)
c1_cols    = list(C1_FEATURE_NAMES)
d_cols     = list(D_FEATURE_NAMES)

BLOCKS = {
    'G':            list(g_cols),
    'A1+B':         a1_cols + b_cols,
    'A1+B+C1':      a1_cols + b_cols + c1_cols,
    'A1+B+C1+D1':   a1_cols + b_cols + c1_cols + d_cols,
    'G+B+C1':       list(g_cols) + b_cols + c1_cols,
}
for n, cols in BLOCKS.items():
    print(f'  {n:<12s} {len(cols):3d} features')

# row arrays
y_row    = df['celltype_label'].to_numpy()
groups_r = df['nucleus_id'].to_numpy()
sess_r   = df['session_key'].to_numpy()
folds_r  = df['gkf_fold_phase3'].to_numpy().astype(np.int8)
ccabs_r  = df['cc_abs'].to_numpy(np.float64)
print(f'\nlong rows: {len(df):,}')


  G            116 features
  A1+B          27 features
  A1+B+C1       31 features
  A1+B+C1+D1    41 features
  G+B+C1       127 features

long rows: 45,832


## 4. Helpers (pipelines, GKF, LOSO, residualization)

In [4]:
def make_pipeline(model_name: str) -> Pipeline:
    if model_name == 'LogReg':
        return Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('scale',  RobustScaler()),
            ('clf',    LogisticRegression(
                penalty='l2', C=1.0, solver='lbfgs', max_iter=400,
                class_weight='balanced', n_jobs=-1, random_state=RANDOM_SEED)),
        ])
    if model_name == 'HGB':
        return Pipeline([
            ('clf', HistGradientBoostingClassifier(
                max_iter=100, max_depth=8, learning_rate=0.05,
                l2_regularization=1.0,
                early_stopping=True, n_iter_no_change=10,
                random_state=RANDOM_SEED)),
        ])
    raise ValueError(model_name)


def _residualize(X_tr, X_te, c_tr, c_te):
    med = np.nanmedian(c_tr)
    c_tr_f = np.where(np.isnan(c_tr), med, c_tr)
    c_te_f = np.where(np.isnan(c_te), med, c_te)
    Xt = X_tr.copy(); Xe = X_te.copy()
    for j in range(X_tr.shape[1]):
        f_tr = Xt[:, j]; m = ~np.isnan(f_tr)
        if m.sum() < 5: continue
        c_fit = c_tr_f[m, 0]; f_fit = f_tr[m]
        cm_, fm_ = c_fit.mean(), f_fit.mean()
        denom = ((c_fit - cm_) ** 2).sum()
        if denom < 1e-12: continue
        b = ((c_fit - cm_) * (f_fit - fm_)).sum() / denom
        a = fm_ - b * cm_
        Xt[:, j] = f_tr - (a + b * c_tr_f[:, 0])
        Xe[:, j] = Xe[:, j] - (a + b * c_te_f[:, 0])
    return Xt, Xe


def gkf_run_long(X, y, groups, folds, model_name,
                 residualize=False, ccabs=None, classes=CLASSES):
    fold_scores = []
    for k in range(N_FOLDS):
        tr = folds != k; te = folds == k
        if tr.sum() == 0 or te.sum() == 0: continue
        if residualize:
            Xt, Xe = _residualize(X[tr], X[te], ccabs[tr].reshape(-1,1), ccabs[te].reshape(-1,1))
        else:
            Xt, Xe = X[tr], X[te]
        pipe = make_pipeline(model_name)
        sw = compute_sample_weight('balanced', y[tr])
        pipe.fit(Xt, y[tr], clf__sample_weight=sw)
        proba = pipe.predict_proba(Xe)
        sc = neuron_level_score(y[te], proba, groups[te], pipe.classes_)
        fold_scores.append(sc)
    return summarize_cv_runs(fold_scores)


def loso_run_long(X, y, groups, sessions, model_name,
                  residualize=False, ccabs=None, classes=CLASSES):
    rows = []
    for sk in sorted(np.unique(sessions).tolist()):
        te = sessions == sk; tr = ~te
        if tr.sum() == 0 or te.sum() == 0: continue
        if len(np.unique(y[tr])) < 2: continue
        if residualize:
            Xt, Xe = _residualize(X[tr], X[te], ccabs[tr].reshape(-1,1), ccabs[te].reshape(-1,1))
        else:
            Xt, Xe = X[tr], X[te]
        pipe = make_pipeline(model_name)
        sw = compute_sample_weight('balanced', y[tr])
        pipe.fit(Xt, y[tr], clf__sample_weight=sw)
        proba = pipe.predict_proba(Xe)
        sc = neuron_level_score(y[te], proba, groups[te], pipe.classes_)
        n_neurons = sc['n_neurons']
        y_neuron = sc['y_neuron_true']
        n_it = int((y_neuron == '6P-IT').sum())
        n_ct = int((y_neuron == '6P-CT').sum())
        rows.append({
            'held_out_scan': sk,
            'n_neurons_test': n_neurons,
            'n_test_IT': n_it, 'n_test_CT': n_ct,
            'minority_n': min(n_it, n_ct),
            'both_classes': (n_it > 0 and n_ct > 0),
            'balanced_accuracy': sc['balanced_accuracy'],
            'macro_f1':          sc['macro_f1'],
            'recall_IT':         sc['per_class_recall'].get('6P-IT', float('nan')),
            'recall_CT':         sc['per_class_recall'].get('6P-CT', float('nan')),
            'cm':                sc['confusion_matrix'].tolist(),
            'n_test_rows':       int(te.sum()),
        })
    return rows


def aggregate_loso(rows, valid_scans):
    df_ps = pd.DataFrame(rows)
    out = {}
    for tag, sub in (('all', df_ps), ('valid', df_ps[df_ps['held_out_scan'].isin(valid_scans)])):
        bals = sub['balanced_accuracy'].dropna().to_numpy(float)
        out[f'bal_acc_{tag}_mean'] = float(bals.mean()) if len(bals) else float('nan')
        out[f'bal_acc_{tag}_std']  = float(bals.std())  if len(bals) else float('nan')
        out[f'n_scans_{tag}']      = int(len(bals))
        for k in ('recall_IT','recall_CT'):
            v = sub[k].dropna().to_numpy(float)
            out[f'{k}_{tag}_mean'] = float(v.mean()) if len(v) else float('nan')
    return out


print('Helpers ready.')


Helpers ready.


## 5. GKF main grid (5 blocks × 2 models)

In [5]:
gkf_summary = []
t0 = time.time()
for block_name, cols in BLOCKS.items():
    X = df[cols].to_numpy(dtype=np.float64)
    for model in ('LogReg','HGB'):
        s = gkf_run_long(X, y_row, groups_r, folds_r, model)
        gkf_summary.append({
            'family':'main','block':block_name,'model':model,'cv':'gkf',
            'n_features': X.shape[1], 'n_rows': int(X.shape[0]),
            **{k:v for k,v in s.items() if k != 'per_class_recall'},
            'recall_IT': s['per_class_recall'].get('6P-IT', float('nan')),
            'recall_CT': s['per_class_recall'].get('6P-CT', float('nan')),
        })
        print(f'[{block_name:<12s} | {model:<6s} | GKF ] '
              f'bal_acc = {s["balanced_accuracy"]:.3f} ± {s["balanced_accuracy_std"]:.3f}  '
              f'| R(IT)={s["per_class_recall"].get("6P-IT", float("nan")):.3f} '
              f'R(CT)={s["per_class_recall"].get("6P-CT", float("nan")):.3f}')
print(f'\nGKF main grid done in {time.time()-t0:.1f}s')


[G            | LogReg | GKF ] bal_acc = 0.562 ± 0.077  | R(IT)=0.646 R(CT)=0.478
[G            | HGB    | GKF ] bal_acc = 0.518 ± 0.062  | R(IT)=0.653 R(CT)=0.384
[A1+B         | LogReg | GKF ] bal_acc = 0.601 ± 0.061  | R(IT)=0.541 R(CT)=0.662
[A1+B         | HGB    | GKF ] bal_acc = 0.580 ± 0.068  | R(IT)=0.632 R(CT)=0.528
[A1+B+C1      | LogReg | GKF ] bal_acc = 0.604 ± 0.066  | R(IT)=0.554 R(CT)=0.653
[A1+B+C1      | HGB    | GKF ] bal_acc = 0.591 ± 0.072  | R(IT)=0.649 R(CT)=0.532
[A1+B+C1+D1   | LogReg | GKF ] bal_acc = 0.606 ± 0.066  | R(IT)=0.549 R(CT)=0.662
[A1+B+C1+D1   | HGB    | GKF ] bal_acc = 0.564 ± 0.050  | R(IT)=0.609 R(CT)=0.520
[G+B+C1       | LogReg | GKF ] bal_acc = 0.552 ± 0.084  | R(IT)=0.641 R(CT)=0.463
[G+B+C1       | HGB    | GKF ] bal_acc = 0.518 ± 0.062  | R(IT)=0.653 R(CT)=0.384

GKF main grid done in 55.3s


## 6. GKF confound baselines + cc_abs-residualized winner

In [6]:
from sklearn.metrics import balanced_accuracy_score, f1_score

# 6.1 Majority class
neuron_df = df.drop_duplicates('nucleus_id').reset_index(drop=True)
y_neu = neuron_df['celltype_label'].to_numpy()
maj = np.unique(y_neu, return_counts=True)[0][np.unique(y_neu, return_counts=True)[1].argmax()]
y_pred_maj = np.full_like(y_neu, maj)
gkf_summary.append({
    'family':'baseline','block':'-','model':'majority','cv':'gkf','n_features':0,'n_rows':int(len(neuron_df)),
    'balanced_accuracy': float(balanced_accuracy_score(y_neu, y_pred_maj)),
    'balanced_accuracy_std': 0.0,
    'macro_f1': float(f1_score(y_neu, y_pred_maj, average='macro', labels=CLASSES, zero_division=0)),
    'macro_f1_std': 0.0,
    'recall_IT': 1.0 if maj=='6P-IT' else 0.0,
    'recall_CT': 1.0 if maj=='6P-CT' else 0.0,
})
print(f'[baseline | majority    | GKF ] bal_acc = {gkf_summary[-1]["balanced_accuracy"]:.3f}  (predicts {maj})')

# 6.2 scan-only (long-row one-hot)
scan_dummies = pd.get_dummies(df['session_key'], prefix='scan').to_numpy(np.float64)
for model in ('LogReg','HGB'):
    s = gkf_run_long(scan_dummies, y_row, groups_r, folds_r, model)
    gkf_summary.append({'family':'baseline','block':'scan-only','model':model,'cv':'gkf',
                        'n_features': scan_dummies.shape[1],'n_rows':int(scan_dummies.shape[0]),
                        **{k:v for k,v in s.items() if k != 'per_class_recall'},
                        'recall_IT': s['per_class_recall'].get('6P-IT', float('nan')),
                        'recall_CT': s['per_class_recall'].get('6P-CT', float('nan'))})
    print(f'[baseline | scan-only   | GKF | {model:<6s}] bal_acc = '
          f'{s["balanced_accuracy"]:.3f} ± {s["balanced_accuracy_std"]:.3f}')

# 6.3 cc_abs-only
ccabs_X = df[['cc_abs']].to_numpy(np.float64)
for model in ('LogReg','HGB'):
    s = gkf_run_long(ccabs_X, y_row, groups_r, folds_r, model)
    gkf_summary.append({'family':'baseline','block':'cc_abs-only','model':model,'cv':'gkf',
                        'n_features':1,'n_rows':int(ccabs_X.shape[0]),
                        **{k:v for k,v in s.items() if k != 'per_class_recall'},
                        'recall_IT': s['per_class_recall'].get('6P-IT', float('nan')),
                        'recall_CT': s['per_class_recall'].get('6P-CT', float('nan'))})
    print(f'[baseline | cc_abs-only | GKF | {model:<6s}] bal_acc = '
          f'{s["balanced_accuracy"]:.3f} ± {s["balanced_accuracy_std"]:.3f}')

# 6.4 GKF winner + residualized (residualized is the L6 PRIMARY headline)
gkf_main_df = pd.DataFrame([r for r in gkf_summary if r['family']=='main'])
winner = gkf_main_df.sort_values('balanced_accuracy', ascending=False).iloc[0]
WINNER_BLOCK = winner['block']; WINNER_MODEL = winner['model']
print(f'\nGKF winner (un-residualized): {WINNER_BLOCK} | {WINNER_MODEL}, '
      f'bal_acc = {winner["balanced_accuracy"]:.3f} ± {winner["balanced_accuracy_std"]:.3f}')

X_w = df[BLOCKS[WINNER_BLOCK]].to_numpy(np.float64)
s_resid_gkf = gkf_run_long(X_w, y_row, groups_r, folds_r, WINNER_MODEL,
                           residualize=True, ccabs=ccabs_r)
gkf_summary.append({'family':'baseline','block':f'{WINNER_BLOCK} (resid cc_abs)','model':WINNER_MODEL,
                    'cv':'gkf','n_features': len(BLOCKS[WINNER_BLOCK]),'n_rows':int(X_w.shape[0]),
                    **{k:v for k,v in s_resid_gkf.items() if k != 'per_class_recall'},
                    'recall_IT': s_resid_gkf['per_class_recall'].get('6P-IT', float('nan')),
                    'recall_CT': s_resid_gkf['per_class_recall'].get('6P-CT', float('nan'))})
print(f'[{WINNER_BLOCK} | {WINNER_MODEL} | GKF | cc_abs-resid] bal_acc = '
      f'{s_resid_gkf["balanced_accuracy"]:.3f} ± {s_resid_gkf["balanced_accuracy_std"]:.3f}'
      f'  (Δ = {s_resid_gkf["balanced_accuracy"] - winner["balanced_accuracy"]:+.3f})')


[baseline | majority    | GKF ] bal_acc = 0.500  (predicts 6P-IT)
[baseline | scan-only   | GKF | LogReg] bal_acc = 0.580 ± 0.009
[baseline | scan-only   | GKF | HGB   ] bal_acc = 0.580 ± 0.009
[baseline | cc_abs-only | GKF | LogReg] bal_acc = 0.542 ± 0.064
[baseline | cc_abs-only | GKF | HGB   ] bal_acc = 0.551 ± 0.053

GKF winner (un-residualized): A1+B+C1+D1 | LogReg, bal_acc = 0.606 ± 0.066
[A1+B+C1+D1 | LogReg | GKF | cc_abs-resid] bal_acc = 0.616 ± 0.049  (Δ = +0.011)


## 7. LOSO grid + baselines + residualized winner

In [7]:
loso_summary = []
loso_perscan = []
t0 = time.time()
for block_name, cols in BLOCKS.items():
    X = df[cols].to_numpy(np.float64)
    for model in ('LogReg','HGB'):
        rows = loso_run_long(X, y_row, groups_r, sess_r, model)
        agg = aggregate_loso(rows, VALID_SCANS)
        for r in rows: loso_perscan.append({'family':'main','block':block_name,'model':model,**r})
        loso_summary.append({'family':'main','block':block_name,'model':model,'cv':'loso',
                             'n_features': X.shape[1], **agg})
        print(f'[{block_name:<12s} | {model:<6s} | LOSO] valid({agg["n_scans_valid"]}) bal_acc = '
              f'{agg["bal_acc_valid_mean"]:.3f} ± {agg["bal_acc_valid_std"]:.3f}')

# LOSO baselines
# 7.1 majority
maj_rows = []
for sk in sorted(np.unique(sess_r).tolist()):
    te = sess_r == sk
    nd = df[te].drop_duplicates('nucleus_id')
    n_it = int((nd['celltype_label']=='6P-IT').sum())
    n_ct = int((nd['celltype_label']=='6P-CT').sum())
    if n_it > 0 and n_ct > 0:
        bal = 0.5  # majority -> recall_IT=1, recall_CT=0
    else:
        bal = None
    maj_rows.append({'held_out_scan':sk,'n_neurons_test':int(len(nd)),
                     'n_test_IT':n_it,'n_test_CT':n_ct,'minority_n':min(n_it,n_ct),
                     'both_classes':(n_it>0 and n_ct>0),
                     'balanced_accuracy':bal,'macro_f1':None,
                     'recall_IT':1.0 if maj=='6P-IT' else 0.0,
                     'recall_CT':1.0 if maj=='6P-CT' else 0.0,
                     'cm':None,'n_test_rows':int(te.sum())})
agg = aggregate_loso(maj_rows, VALID_SCANS)
loso_summary.append({'family':'baseline','block':'-','model':'majority','cv':'loso','n_features':0,**agg})
for r in maj_rows: loso_perscan.append({'family':'baseline','block':'-','model':'majority',**r})
print(f'\n[baseline | majority    | LOSO] valid bal_acc = {agg["bal_acc_valid_mean"]:.3f}')

# 7.2 scan-only (must be ~0.5 under LOSO)
for model in ('LogReg','HGB'):
    rows = loso_run_long(scan_dummies, y_row, groups_r, sess_r, model)
    agg = aggregate_loso(rows, VALID_SCANS)
    loso_summary.append({'family':'baseline','block':'scan-only','model':model,'cv':'loso',
                         'n_features': scan_dummies.shape[1], **agg})
    for r in rows: loso_perscan.append({'family':'baseline','block':'scan-only','model':model,**r})
    print(f'[baseline | scan-only   | LOSO | {model:<6s}] valid bal_acc = '
          f'{agg["bal_acc_valid_mean"]:.3f} ± {agg["bal_acc_valid_std"]:.3f}  '
          f'(must be ~0.5)')

# 7.3 cc_abs-only LOSO
for model in ('LogReg','HGB'):
    rows = loso_run_long(ccabs_X, y_row, groups_r, sess_r, model)
    agg = aggregate_loso(rows, VALID_SCANS)
    loso_summary.append({'family':'baseline','block':'cc_abs-only','model':model,'cv':'loso',
                         'n_features':1,**agg})
    for r in rows: loso_perscan.append({'family':'baseline','block':'cc_abs-only','model':model,**r})
    print(f'[baseline | cc_abs-only | LOSO | {model:<6s}] valid bal_acc = '
          f'{agg["bal_acc_valid_mean"]:.3f} ± {agg["bal_acc_valid_std"]:.3f}')

# 7.4 cc_abs-residualized winner LOSO (PRIMARY headline for L6 IT/CT)
rows = loso_run_long(X_w, y_row, groups_r, sess_r, WINNER_MODEL,
                     residualize=True, ccabs=ccabs_r)
agg_resid_loso = aggregate_loso(rows, VALID_SCANS)
loso_summary.append({'family':'baseline','block':f'{WINNER_BLOCK} (resid cc_abs)',
                     'model':WINNER_MODEL,'cv':'loso',
                     'n_features': len(BLOCKS[WINNER_BLOCK]), **agg_resid_loso})
for r in rows: loso_perscan.append({'family':'baseline',
                                    'block':f'{WINNER_BLOCK} (resid cc_abs)',
                                    'model':WINNER_MODEL,**r})
print(f'\n[PRIMARY] {WINNER_BLOCK} | {WINNER_MODEL} | LOSO | cc_abs-resid : '
      f'valid = {agg_resid_loso["bal_acc_valid_mean"]:.3f} ± {agg_resid_loso["bal_acc_valid_std"]:.3f}'
      f'  (n={agg_resid_loso["n_scans_valid"]}, in {time.time()-t0:.1f}s total)')


/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


[G            | LogReg | LOSO] valid(6) bal_acc = 0.534 ± 0.058


/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


[G            | HGB    | LOSO] valid(6) bal_acc = 0.522 ± 0.032


/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


...

[baseline | cc_abs-only | LOSO | LogReg] valid bal_acc = 0.507 ± 0.089


/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


[baseline | cc_abs-only | LOSO | HGB   ] valid bal_acc = 0.556 ± 0.039


/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")



[PRIMARY] A1+B+C1+D1 | LogReg | LOSO | cc_abs-resid : valid = 0.570 ± 0.066  (n=6, in 88.8s total)


## 8. Summary tables

In [8]:
gkf_df  = pd.DataFrame(gkf_summary)
loso_df = pd.DataFrame(loso_summary)
gkf_df['bal_acc_str'] = gkf_df.apply(
    lambda r: f"{r['balanced_accuracy']:.3f} ± {r.get('balanced_accuracy_std',0):.3f}", axis=1)
loso_df['valid_str'] = loso_df.apply(
    lambda r: f"{r['bal_acc_valid_mean']:.3f} ± {r['bal_acc_valid_std']:.3f} (n={int(r['n_scans_valid'])})", axis=1)
loso_df['all_str']   = loso_df.apply(
    lambda r: f"{r['bal_acc_all_mean']:.3f} ± {r['bal_acc_all_std']:.3f} (n={int(r['n_scans_all'])})", axis=1)

print('=== L6 IT/CT — GKF MAIN GRID (long-row + neuron-level prob agg) ===')
print(gkf_df[gkf_df['family']=='main']
      .sort_values('balanced_accuracy', ascending=False)
      [['block','model','n_features','bal_acc_str','recall_IT','recall_CT','macro_f1']]
      .to_string(index=False))
print()
print('=== L6 IT/CT — GKF BASELINES ===')
print(gkf_df[gkf_df['family']=='baseline']
      [['block','model','n_features','bal_acc_str','recall_IT','recall_CT','macro_f1']]
      .to_string(index=False))
print()
print('=== L6 IT/CT — LOSO MAIN GRID ===')
print(loso_df[loso_df['family']=='main']
      .sort_values('bal_acc_valid_mean', ascending=False)
      [['block','model','n_features','valid_str','all_str',
        'recall_IT_valid_mean','recall_CT_valid_mean']]
      .to_string(index=False))
print()
print('=== L6 IT/CT — LOSO BASELINES ===')
print(loso_df[loso_df['family']=='baseline']
      [['block','model','n_features','valid_str','all_str',
        'recall_IT_valid_mean','recall_CT_valid_mean']]
      .to_string(index=False))


=== L6 IT/CT — GKF MAIN GRID (long-row + neuron-level prob agg) ===
     block  model  n_features   bal_acc_str  recall_IT  recall_CT  macro_f1
A1+B+C1+D1 LogReg          41 0.606 ± 0.066   0.549419   0.661776  0.579193
   A1+B+C1 LogReg          31 0.604 ± 0.066   0.553964   0.653080  0.578893
      A1+B LogReg          27 0.601 ± 0.061   0.540692   0.661776  0.573832
   A1+B+C1    HGB          31 0.591 ± 0.072   0.649417   0.531936  0.584361
      A1+B    HGB          27 0.580 ± 0.068   0.631815   0.527529  0.573010
A1+B+C1+D1    HGB          41 0.564 ± 0.050   0.609161   0.519629  0.556189
         G LogReg         116 0.562 ± 0.077   0.646003   0.478257  0.558333
    G+B+C1 LogReg         127 0.552 ± 0.084   0.641381   0.462873  0.548016
         G    HGB         116 0.518 ± 0.062   0.652668   0.383507  0.512807
    G+B+C1    HGB         127 0.518 ± 0.062   0.652668   0.383507  0.512807

=== L6 IT/CT — GKF BASELINES ===
                    block    model  n_features   bal_acc_str  

## 9. Per-scan winner table (LOSO)

In [9]:
perscan = pd.DataFrame(loso_perscan)
ps_w = perscan[(perscan['family']=='main') &
               (perscan['block']==WINNER_BLOCK) &
               (perscan['model']==WINNER_MODEL)].copy()
ps_w['is_valid'] = ps_w['held_out_scan'].isin(VALID_SCANS)
ps_w = ps_w.sort_values(['is_valid','balanced_accuracy'], ascending=[False, False])
print(f'WINNER (un-residualized): {WINNER_BLOCK} | {WINNER_MODEL}')
print(ps_w[['held_out_scan','is_valid','n_neurons_test','n_test_IT','n_test_CT',
            'minority_n','balanced_accuracy','recall_IT','recall_CT']].to_string(index=False))

ps_r = perscan[(perscan['family']=='baseline') &
               (perscan['block']==f'{WINNER_BLOCK} (resid cc_abs)') &
               (perscan['model']==WINNER_MODEL)].copy()
ps_r['is_valid'] = ps_r['held_out_scan'].isin(VALID_SCANS)
ps_r = ps_r.sort_values(['is_valid','balanced_accuracy'], ascending=[False, False])
print()
print(f'WINNER (cc_abs-residualized) — PRIMARY result for L6 IT/CT:')
print(ps_r[['held_out_scan','is_valid','n_neurons_test','n_test_IT','n_test_CT',
            'balanced_accuracy','recall_IT','recall_CT']].to_string(index=False))


WINNER (un-residualized): A1+B+C1+D1 | LogReg
held_out_scan  is_valid  n_neurons_test  n_test_IT  n_test_CT  minority_n  balanced_accuracy  recall_IT  recall_CT
          5_7      True              81         57         24          24           0.664474   0.578947   0.750000
          5_6      True              96         46         50          46           0.592174   0.304348   0.880000
          6_6      True              34         23         11          11           0.577075   0.608696   0.545455
          6_7      True              34         22         12          12           0.568182   0.636364   0.500000
          6_2      True              36         27          9           9           0.481481   0.407407   0.555556
          6_4      True              55         41         14          14           0.471254   0.585366   0.357143
          4_7     False               1          0          1           0           0.000000        NaN   0.000000

WINNER (cc_abs-residualized) — PR

## 10. Save

In [10]:
gkf_df.to_parquet(GKF_OUT, index=False)
print(f'wrote {GKF_OUT}  ({GKF_OUT.stat().st_size/1024:.1f} KB, {len(gkf_df)} rows)')
loso_df.to_parquet(LOSO_OUT, index=False)
print(f'wrote {LOSO_OUT}  ({LOSO_OUT.stat().st_size/1024:.1f} KB, {len(loso_df)} rows)')

ps_save = perscan.copy()
ps_save['cm'] = ps_save['cm'].apply(lambda v: json.dumps(v) if v is not None else None)
ps_save.to_parquet(PERSCAN_OUT, index=False)
print(f'wrote {PERSCAN_OUT}  ({PERSCAN_OUT.stat().st_size/1024:.1f} KB, {len(ps_save)} rows)')

w_loso = loso_df.loc[(loso_df['family']=='main') &
                     (loso_df['block']==WINNER_BLOCK) &
                     (loso_df['model']==WINNER_MODEL)].iloc[0]
winner_meta = {
    'phase':         'phase3_step8_v2_longrow',
    'task':          'L6_IT_vs_CT',
    'protocol':      'long-row + per-neuron probability aggregation (WORKFLOW §3.6)',
    'winner_block':  WINNER_BLOCK,
    'winner_model':  WINNER_MODEL,
    'gkf_winner_balanced_accuracy_mean': float(winner['balanced_accuracy']),
    'gkf_residualized_balanced_accuracy_mean': float(s_resid_gkf['balanced_accuracy']),
    'loso_winner_valid_balanced_accuracy_mean': float(w_loso['bal_acc_valid_mean']),
    'loso_residualized_valid_balanced_accuracy_mean': float(agg_resid_loso['bal_acc_valid_mean']),
    'loso_residualized_valid_balanced_accuracy_std':  float(agg_resid_loso['bal_acc_valid_std']),
    'n_valid_loso_scans': int(agg_resid_loso['n_scans_valid']),
    'note': 'Per PHASE3_PLANNING §3, the cc_abs-residualized LOSO valid mean is the PRIMARY headline.',
}
with open(WINNER_OUT,'w') as f:
    json.dump(winner_meta, f, indent=2)
print(f'wrote {WINNER_OUT}')


wrote /Users/katiarusso/Documents/VS CODE/Classes/Neuroscience/Final Project/data/processed/results/phase3_l6_it_ct_runs.parquet  (9.3 KB, 16 rows)
wrote /Users/katiarusso/Documents/VS CODE/Classes/Neuroscience/Final Project/data/processed/results/phase3_l6_it_ct_loso.parquet  (11.7 KB, 16 rows)
wrote /Users/katiarusso/Documents/VS CODE/Classes/Neuroscience/Final Project/data/processed/results/phase3_l6_it_ct_loso_perscan.parquet  (11.7 KB, 112 rows)
wrote /Users/katiarusso/Documents/VS CODE/Classes/Neuroscience/Final Project/data/processed/results/phase3_l6_it_ct_winner.json


## 11. Step-8 summary

In [11]:
print('=== PHASE 3 STEP 8 (REBUILD) — L6 IT/CT ===')
print(f'protocol         : long-row + per-neuron probability aggregation (WORKFLOW §3.6)')
print(f'population       : 6P-IT={int((y_neu=="6P-IT").sum())}, 6P-CT={int((y_neu=="6P-CT").sum())}')
print(f'unique scans     : {df["session_key"].nunique()}')
print(f'LOSO valid scans : {len(VALID_SCANS)} -> {VALID_SCANS}')
print()
print(f'GKF winner       : {WINNER_BLOCK} | {WINNER_MODEL}')
print(f'  GKF bal_acc                : {winner["balanced_accuracy"]:.3f} ± {winner["balanced_accuracy_std"]:.3f}')
print(f'  GKF cc_abs-resid           : {s_resid_gkf["balanced_accuracy"]:.3f}  '
      f'(Δ = {s_resid_gkf["balanced_accuracy"] - winner["balanced_accuracy"]:+.3f})')
print()
print(f'LOSO winner valid           : {w_loso["bal_acc_valid_mean"]:.3f} ± {w_loso["bal_acc_valid_std"]:.3f} '
      f'(n={int(w_loso["n_scans_valid"])})')
print(f'LOSO cc_abs-resid PRIMARY   : {agg_resid_loso["bal_acc_valid_mean"]:.3f} ± '
      f'{agg_resid_loso["bal_acc_valid_std"]:.3f}  '
      f'(Δ = {agg_resid_loso["bal_acc_valid_mean"]-w_loso["bal_acc_valid_mean"]:+.3f})')
print()
maj_v = loso_df.loc[loso_df["model"]=="majority","bal_acc_valid_mean"].iloc[0]
sc_lr = loso_df.loc[(loso_df['block']=='scan-only')&(loso_df['model']=='LogReg'),'bal_acc_valid_mean'].iloc[0]
sc_hgb = loso_df.loc[(loso_df['block']=='scan-only')&(loso_df['model']=='HGB'),'bal_acc_valid_mean'].iloc[0]
ca_lr = loso_df.loc[(loso_df['block']=='cc_abs-only')&(loso_df['model']=='LogReg'),'bal_acc_valid_mean'].iloc[0]
ca_hgb = loso_df.loc[(loso_df['block']=='cc_abs-only')&(loso_df['model']=='HGB'),'bal_acc_valid_mean'].iloc[0]
print(f'LOSO majority           : {maj_v:.3f}')
print(f'LOSO scan-only LR/HGB   : {sc_lr:.3f} / {sc_hgb:.3f}  (must be ~0.5; integrity check)')
print(f'LOSO cc_abs-only LR/HGB : {ca_lr:.3f} / {ca_hgb:.3f}')
print()
print('PRIMARY reportable for L6 IT/CT (cc_abs-residualized LOSO valid):')
print(f'  bal_acc = {agg_resid_loso["bal_acc_valid_mean"]:.3f} ± {agg_resid_loso["bal_acc_valid_std"]:.3f}')
print(f'  R(IT)   = {agg_resid_loso["recall_IT_valid_mean"]:.3f}')
print(f'  R(CT)   = {agg_resid_loso["recall_CT_valid_mean"]:.3f}')


=== PHASE 3 STEP 8 (REBUILD) — L6 IT/CT ===
protocol         : long-row + per-neuron probability aggregation (WORKFLOW §3.6)
population       : 6P-IT=216, 6P-CT=121
unique scans     : 7
LOSO valid scans : 6 -> ['6_4', '6_7', '6_2', '5_7', '6_6', '5_6']

GKF winner       : A1+B+C1+D1 | LogReg
  GKF bal_acc                : 0.606 ± 0.066
  GKF cc_abs-resid           : 0.616  (Δ = +0.011)

LOSO winner valid           : 0.559 ± 0.066 (n=6)
LOSO cc_abs-resid PRIMARY   : 0.570 ± 0.066  (Δ = +0.011)

LOSO majority           : 0.500
LOSO scan-only LR/HGB   : 0.500 / 0.500  (must be ~0.5; integrity check)
LOSO cc_abs-only LR/HGB : 0.507 / 0.556

PRIMARY reportable for L6 IT/CT (cc_abs-residualized LOSO valid):
  bal_acc = 0.570 ± 0.066
  R(IT)   = 0.583
  R(CT)   = 0.557
